In [2]:
import os
import time
import numpy as np
from tqdm import tqdm

import pybullet as p
import pybullet_data
from surrol.utils.pybullet_utils import (
    step,
    get_joints,
    get_link_name,
    reset_camera,
)
from surrol.robots.psm import Psm

pybullet build time: Oct 15 2024 14:38:58


In [3]:
scaling = 1.

p.connect(p.GUI)
# p.connect(p.DIRECT)
p.setGravity(0, 0, -9.81)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
reset_camera(yaw=90, pitch=-15, dist=0.9*scaling)

# p.setPhysicsEngineParameter(contactBreakingThreshold=0.002)

In [4]:
p.loadURDF("plane.urdf", [0, 0, -0.001], globalScaling=1)

psm = Psm((0, 0, 0.1524),
          p.getQuaternionFromEuler((0, 0, -90/180*np.pi)), 
          scaling=scaling)
psm.reset_joint((0, 0, 0.10, 0, 0, 0))

# psm = Psm((0.05, 0.24, 0.8524),
#           p.getQuaternionFromEuler((0, 0, np.deg2rad(-(90+20)))), 
#           scaling=scaling)
# psm.reset_joint((0, 0, 0.10, 0, 0, 0))

# for info in get_joints_info(psm.body, psm.joints):
#     print(info)

(0, 0, 0.1, 0, 0, 0)

In [5]:
joints = get_joints(psm.body)
print("There are {} joints.\n".format(len(joints)))

for i in range(0, len(joints)):
    print(get_link_name(psm.body, i))

There are 16 joints.

psm_yaw_link
psm_pitch_end_link
psm_main_insertion_link
psm_tool_roll_link
psm_tool_pitch_link
psm_tool_yaw_link
psm_tool_gripper1_link
psm_tool_gripper2_link
psm_tool_tip_link
psm_pitch_back_link
psm_pitch_bottom_link
psm_pitch_top_link
psm_pitch_front_link
psm_remote_center_link
psm_main_insertion_link_2
psm_main_insertion_link_3


In [ ]:
# continously run
p.setRealTimeSimulation(1)

while True:
    p.setGravity(0, 0, -10)
    time.sleep(0.01)

## Forward Kinematics

In [6]:
# Test with predefined pose
# original joint position; 0.jpg

psm.move_joint([-0.52359879, 0., 0.12, 0., 0., 0.])
step(0.5)
print(psm.get_current_position())

# Read from dVRK (get_position_current)
# [[-0.0077    0.8686   -0.4955   -0.0567]
#  [ 0.9999    0.0001   -0.0154   -0.0002]
#  [-0.0133   -0.4956   -0.8685   -0.0982]
#  [      0         0         0    1.0000]]

# previously compute
# [[-6.12323415e-17  8.66025396e-01 -5.00000013e-01 -5.67500001e-02]
#  [ 1.00000000e+00 -3.06161708e-17 -1.75493441e-16 -1.35258490e-17]
#  [-1.67289863e-16 -5.00000013e-01 -8.66025396e-01 -9.82938802e-02]
#  [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]

[[-4.21469382e-08  8.66026646e-01 -4.99997849e-01 -5.67497276e-02]
 [ 1.00000000e+00  7.30004289e-08  4.21469382e-08  0.00000000e+00]
 [ 7.30004289e-08 -4.99997849e-01 -8.66026646e-01 -9.82939601e-02]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [7]:
# simple joint position 2; 3.jpg

psm.move_joint([-1.0471975511965976, 0.17453292519943295, 0.1, 0.7853981633974483, 0, 0])
step(0.5)
print(psm.get_current_position())

# [[ 0.24721603  0.45989072 -0.85286855 -0.07974321]
#  [ 0.69636423 -0.69636426 -0.17364817 -0.0162361 ]
#  [-0.67376636 -0.55097853 -0.49240385 -0.04603976]
#  [ 0.          0.          0.          1.        ]]

[[ 0.24721678  0.45989241 -0.85286742 -0.07974317]
 [ 0.69636627 -0.69636236 -0.17364755 -0.01623607]
 [-0.67376396 -0.55097952 -0.49240602 -0.04604   ]
 [ 0.          0.          0.          1.        ]]


In [8]:
# simple joint position 3; 4.jpg

psm.move_joint([-1.0471975511965976, 0.17453292519943295, 0.1, 0.7853981633974483, 0.5235987755982988, 0.2617993877991494])
step(0.5)
print(psm.get_current_position())

# [[-0.21233893  0.66737769 -0.71380613 -0.07982825]
#  [ 0.51624502 -0.54359788 -0.66180997 -0.01919286]
#  [-0.82970071 -0.50902688 -0.22910342 -0.0423738 ]
#  [ 0.          0.          0.          1.        ]]

[[-0.2124073   0.66740572 -0.71375959 -0.07982791]
 [ 0.51633149 -0.54347994 -0.66183937 -0.01919338]
 [-0.8296294  -0.50911606 -0.22916348 -0.04237411]
 [ 0.          0.          0.          1.        ]]


## Open/Close Jaw

In [9]:
# open jaw test

psm.open_jaw()
step(0.5)
print(psm.get_current_position())
print(np.rad2deg(psm.get_current_jaw_position()))

[[-0.21237824  0.66739417 -0.71377903 -0.07982806]
 [ 0.51629308 -0.54353075 -0.66182761 -0.01919317]
 [-0.82966074 -0.50907696 -0.22913689 -0.04237397]
 [ 0.          0.          0.          1.        ]]
79.9997773778801


In [10]:
# close jaw test

psm.close_jaw()
step(0.5)
print(np.rad2deg(psm.get_current_jaw_position()))

0.0003331028347109802


## Inverse Kinematics

In [11]:
# Inverse kinematics test
# simple coord position 0; 5.jpg
pose = np.array([
    [ 4.33293348e-13,  1.00000000e+00,  4.52526905e-13, -1.33425265e-17],
    [ 1.00000000e+00, -4.33293348e-13,  9.95143306e-14, -1.12816941e-17],
    [ 9.95143306e-14,  4.52526905e-13, -1.00000000e+00, -1.13499997e-01],
    [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  1.00000000e+00]
])
psm.move(pose)
step(0.5)
print(psm.get_current_joint_position())
print(psm.get_current_position())

# [ 0.00000000e+00  8.65586272e-15  1.19999997e-01  4.33293348e-13
#  -1.08357767e-13  4.52504218e-13]

[-0.06530592397076965, -0.008559137916445773, 0.11974462142838711, 5.59331791888454e-06, 0.008616060540530308, 0.06572184353979388]
[[ 2.15770095e-06  9.99999914e-01  4.15384702e-04 -7.39010656e-03]
 [ 9.99999998e-01 -2.13383341e-06 -5.74593224e-05  8.90851021e-04]
 [-5.74584311e-05  4.15384825e-04 -9.99999912e-01 -1.12999406e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [12]:
# simple coord position 1; 6.jpg
pose = np.array([
    [ 4.33293348e-13,  1.00000000e+00,  4.52526905e-13,  5.00000000e-02],
    [ 1.00000000e+00, -4.33293348e-13,  9.95143306e-14, -1.12816941e-17],
    [ 9.95143306e-14,  4.52526905e-13, -1.00000000e+00, -1.13499997e-01],
    [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  1.00000000e+00]
])
psm.move(pose)
step(0.5)
print(psm.get_current_joint_position())
print(psm.get_current_position())

# [ 4.14949688e-01 -7.10542736e-15  1.30525197e-01  4.36665965e-13
#   9.00240590e-14 -4.14949688e-01]

[0.39012354554254314, -0.0007022692707624029, 0.1304851206330656, 4.865242702533231e-07, 0.0007076492666705313, -0.3899323994607313]
[[ 2.50385066e-06  9.99999982e-01  1.91092483e-04  4.71518375e-02]
 [ 1.00000000e+00 -2.50293946e-06 -4.76861073e-06  8.06152821e-05]
 [-4.76813235e-06  1.91092495e-04 -9.99999982e-01 -1.14669118e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


## JIGSAWS Kinematics Record

In [14]:
# jigsaws test
# import kinematics data transformed from JIGSAWS 
# (https://cirl.lcsr.jhu.edu/research/hmm/datasets/jigsaws_release/)

joint_values = np.load('qs_jigsaws.npy')
psm.close_jaw()
step(1)

start_time = time.time()
for i in tqdm(range(len(joint_values))):
    psm.move_joint(joint_values[i])
    psm.close_jaw()
    step(0.5)
    _ = p.getCameraImage(128, 128)

end_time = time.time()
print("Used time: {:.4f}".format(end_time - start_time))

100%|███████████████████████████████████████| 3420/3420 [01:16<00:00, 44.75it/s]

Used time: 76.4230



X connection to :0 broken (explicit kill or server shutdown).


startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=NVIDIA Corporation
GL_RENDERER=NVIDIA GeForce RTX 3060 Laptop GPU/PCIe/SSE2
GL_VERSION=3.3.0 NVIDIA 550.107.02
GL_SHADING_LANGUAGE_VERSION=3.30 NVIDIA via Cg compiler
pthread_getconcurrency()=0
Version = 3.3.0 NVIDIA 550.107.02
Vendor = NVIDIA Corporation
Renderer = NVIDIA GeForce RTX 3060 Laptop GPU/PCIe/SSE2
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started
ven = NVIDIA Corporation
ven = NVIDIA Corporation


In [ ]:
_ = p.getCameraImage(512, 512)